# Goals

* Edit the SQL database used for scBaseCamp

In [27]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [28]:
import os
import pandas as pd
from pypika import Query, Table, Field, Column, Criterion

In [29]:
from SRAgent.db.connect import db_connect
from SRAgent.db.upsert import db_upsert
from SRAgent.db.utils import db_list_tables, db_glimpse_tables, db_get_table, execute_query
from SRAgent.db.get import db_find_srx
from SRAgent.db.create import create_table, create_table_router

In [30]:
# set to production
os.environ['DYNACONF'] = 'prod'

# Summary

In [31]:
# list tables
with db_connect() as conn:
    print("\n".join(db_list_tables(conn)))

screcounter_star_results
eval
scbasecamp_metadata
screcounter_trace
srx_srr
srx_metadata
screcounter_log
screcounter_star_params


# Updates

## Update values

### Example

In [26]:
# list all records for a feature in scbasecamp_metadata
tbl = Table("scbasecamp_metadata")
stmt = Query \
    .from_(tbl) \
    .select("*") \
    .where(tbl.feature_type == "Gene")

with db_connect() as conn:
    df = pd.read_sql(str(stmt), conn)
df

,entrez_id,srx_accession,feature_type,file_path,obs_count,lib_prep,tech_10x,cell_prep,organism,tissue,disease,perturbation,cell_line,czi_collection_id,czi_collection_name,created_at,updated_at
0,26779693,SRX19498727,Gene,gs://arc-ctc-nextflow/gcp-loader/output2/h5ad/...,4763,10x_Genomics,3_prime_gex,single_nucleus,thale_cress,other,not specified,not specified,not applicable,NaN,NaN,2025-02-23 15:05:19.510071,2025-03-08 01:50:19.616266
1,28156605,SRX20726185,Gene,gs://arc-ctc-nextflow/gcp-loader/output2/h5ad/...,5675,10x_Genomics,3_prime_gex,single_cell,fruit_fly,other,not specified,not specified,not specified; sample is from third instar larvae,NaN,NaN,2025-02-23 15:05:20.980677,2025-03-08 01:50:19.616266
2,32675995,SRX24361339,Gene,gs://arc-ctc-nextflow/gcp-loader/output2/h5ad/...,5834,10x_Genomics,3_prime_gex,single_cell,mouse,brain,"neuroinflammation, Parkinson's disease context",Lipopolysaccharide (LPS),not specified,NaN,NaN,2025-02-23 15:05:24.990988,2025-03-08 01:50:19.616266


In [25]:
# update all of the values
tbl = Table("scbasecamp_metadata")
stmt = Query \
    .update(tbl) \
    .set(tbl.lib_prep, "10x_Genomics") \
    .where(tbl.feature_type == "Gene")

with db_connect() as conn:
    curr = conn.cursor()
    curr.execute(str(stmt))
    conn.commit()

### Update to 10x_Genomics

In [39]:
# list all records for a feature in scbasecamp_metadata
tbl = Table("scbasecamp_metadata")
stmt = Query \
    .from_(tbl) \
    .select("*") \
    .where(tbl.lib_prep != "10x_Genomics")

with db_connect() as conn:
    df = pd.read_sql(str(stmt), conn)
df.shape

(0, 17)

In [40]:
df["lib_prep"].value_counts()

Series([], Name: count, dtype: int64)

In [37]:
df["tech_10x"].value_counts()

tech_10x
not_applicable    866
Name: count, dtype: int64

In [ ]:
# update all of the values
tbl = Table("scbasecamp_metadata")
stmt = Query \
    .update(tbl) \
    .set(tbl.lib_prep, "10x_Genomics") \
    .set(tbl.tech_10x, "other") \
    .where(tbl.lib_prep != "10x_Genomics")

# with db_connect() as conn:
#     curr = conn.cursor()
#     curr.execute(str(stmt))
#     conn.commit()

## Edit column name

In [10]:
# with db_connect() as conn:
#     with conn.cursor() as cur:
#         cur.execute("ALTER TABLE scbasecamp_metadata ADD COLUMN IF NOT EXISTS feature_type VARCHAR(30)")
#         conn.commit()

## Delete target records

In [ ]:
# delete target records
# target_orgs = ["Gallus gallus", "Gorilla gorilla", "Heterocephalus glaber"]
# feat_type = "GeneFull_Ex50pAS"
# os.environ['DYNACONF'] = 'prod'
# tbl = Table("scbasecamp_metadata")
# stmt = Query \
#     .from_(tbl) \
#     .delete() \
#     .where(tbl.feature_type == feat_type) \
#     .where(tbl.organism.isin(target_orgs))

# with db_connect() as conn:
#     with conn.cursor() as cur:
#         cur.execute(str(stmt))
#         conn.commit()